# Uso reproducible de las utilidades del disdrómetro

Los ejemplos procesan los CSV por bloques para no conservar los siete años y las 440 clases simultáneamente en RAM. Si faltan los datos, las celdas informan qué carpeta debe completarse sin producir una traza de error.

In [1]:
from collections import defaultdict
import pandas as pd
import utilidades_disdrometro as disd

In [2]:
# Listar las entradas sin depender de una ruta absoluta del equipo.
if disd.DEFAULT_RESULTS_DIR.exists():
    archivos = disd.list_disdro_files()
else:
    archivos = []
    print(f"Falta la carpeta de datos: {disd.DEFAULT_RESULTS_DIR}")
archivos

[WindowsPath('E:/Climatologia/T2 Disdrometro/resultados/Disdrodata_417_2019.csv'),
 WindowsPath('E:/Climatologia/T2 Disdrometro/resultados/Disdrodata_417_2020.csv'),
 WindowsPath('E:/Climatologia/T2 Disdrometro/resultados/Disdrodata_417_2021.csv'),
 WindowsPath('E:/Climatologia/T2 Disdrometro/resultados/Disdrodata_417_2022.csv'),
 WindowsPath('E:/Climatologia/T2 Disdrometro/resultados/Disdrodata_417_2023.csv'),
 WindowsPath('E:/Climatologia/T2 Disdrometro/resultados/Disdrodata_417_2024.csv'),
 WindowsPath('E:/Climatologia/T2 Disdrometro/resultados/Disdrodata_417_2025.csv')]

In [3]:
# Obtener una muestra con todas las clases, sin combinar el conjunto completo.
columnas_espectro = [
    "fecha_hora",
    "cliente",
    disd.PRECIPITATION_COLUMN,
    *disd.PRECIPITATION_SPECTRUM_COLUMNS,
]
muestra_lluvia = None
if archivos:
    first = next(
        disd.iter_disdro_chunks(
            precipitation_only=True,
            chunksize=5_000,
            usecols=columnas_espectro,
        ),
        None,
    )
    if first is not None:
        year, muestra_lluvia = first
        print(f"Muestra del año {year}: {len(muestra_lluvia)} filas filtradas")
muestra_lluvia.head() if muestra_lluvia is not None else None

Muestra del año 2019: 1928 filas filtradas


,fecha_hora,cliente,var8,var42,var43,var44,var45,var46,var47,var48,...,var472,var473,var474,var475,var476,var477,var478,var479,var480,var481
0,2019-12-12 17:23:00,417,0.038,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2019-12-12 17:28:00,417,0.298,0.0,0.0,0.0,0.0,1.0,3.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2019-12-12 17:30:00,417,0.605,0.0,0.0,1.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2019-12-12 17:31:00,417,0.467,0.0,0.0,0.0,3.0,0.0,1.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2019-12-12 17:32:00,417,0.301,0.0,0.0,0.0,0.0,1.0,3.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
# Contar filas lluviosas por año conservando solo un bloque a la vez.
filas_por_anio = defaultdict(int)
if archivos:
    for year, chunk in disd.iter_disdro_chunks(
        precipitation_only=True,
        chunksize=10_000,
        usecols=["fecha_hora", "cliente", disd.PRECIPITATION_COLUMN],
    ):
        filas_por_anio[year] += len(chunk)
pd.Series(dict(sorted(filas_por_anio.items())), name="filas_con_precipitacion")

2019     3549
2020    68111
2021    68737
2022    81654
2023    51071
2024    56859
2025    14598
Name: filas_con_precipitacion, dtype: int64

In [5]:
# Cargar todos los años con solo tres columnas mantiene un uso moderado de RAM.
por_anio = (
    disd.read_all_disdro_files(
        precipitation_only=True,
        chunksize=10_000,
        usecols=["fecha_hora", "cliente", disd.PRECIPITATION_COLUMN],
    )
    if archivos
    else {}
)
df_2022 = por_anio.get(2022)
df_2022.head() if df_2022 is not None else None

,fecha_hora,cliente,var8
0,2022-01-01 02:59:00,417,0.017
1,2022-01-01 03:00:00,417,0.117
2,2022-01-01 03:01:00,417,0.371
3,2022-01-01 03:02:00,417,0.385
4,2022-01-01 03:03:00,417,0.598
